# 02: Output-Level Evaluation

**Thomson Reuters | Agentic AI Evaluation Bootcamp**

In Notebook 01 we ran the agent on one question and eyeballed the answer.
That doesn't scale. Here we run a **systematic evaluation** using an LLM-as-judge.

## What You'll Do

1. Upload a small DeepSearchQA subset to Langfuse as a named dataset
2. Run the agent on every item using `run_experiment`
3. Score each answer with the **DeepSearchQA grader** (Precision / Recall / F1)
4. Inspect results in a DataFrame and in Langfuse

## The Grader

The DeepSearchQA grader is an **LLM-as-judge** that compares the agent's answer to ground truth:

| Metric | Meaning |
|--------|---------|
| **Precision** | Of the items the agent said, how many were correct? |
| **Recall** | Of the correct items, how many did the agent find? |
| **F1** | Harmonic mean of Precision and Recall |
| **Outcome** | `fully_correct`, `correct_with_extraneous`, `partially_correct`, `fully_incorrect` |

## Prerequisites

- Notebook 01 completed (Langfuse tracing confirmed working)
- All keys in `~/.env`

## 1. Setup & Configuration

In [2]:
import json
import os
import tempfile
from pathlib import Path
from typing import Any

import pandas as pd
from aieng.agent_evals.async_client_manager import AsyncClientManager
from aieng.agent_evals.evaluation import run_experiment
from aieng.agent_evals.evaluation.graders.config import LLMRequestConfig
from aieng.agent_evals.knowledge_qa import DeepSearchQADataset, KnowledgeGroundedAgent
from aieng.agent_evals.knowledge_qa.deepsearchqa_grader import DeepSearchQAResult, evaluate_deepsearchqa_async
from aieng.agent_evals.langfuse import init_tracing, upload_dataset_to_langfuse
from dotenv import load_dotenv
from IPython.display import HTML, display
from langfuse.experiment import Evaluation
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

# Set working directory to repo root
if Path('').absolute().name == 'eval-agents':
    print(f'Working directory: {Path("").absolute()}')
else:
    os.chdir(Path('').absolute().parent.parent)
    print(f'Working directory set to: {Path("").absolute()}')

load_dotenv(verbose=True)
init_tracing()
console = Console(width=100)

# ── Configuration ──────────────────────────────────────────────────────────────
CATEGORY       = 'History'   # DeepSearchQA category
NUM_SAMPLES    = 5                       # Keep small for Build Day (each ~30-60s)
DATASET_NAME   = 'TR-DeepSearchQA'      # Name of dataset in Langfuse
EXPERIMENT_NAME = 'tr-baseline-v1'      # Change this for each new run
# ───────────────────────────────────────────────────────────────────────────────

console.print(Panel(
    f'[bold]Category:[/bold]   {CATEGORY}\n'
    f'[bold]Samples:[/bold]    {NUM_SAMPLES}\n'
    f'[bold]Dataset:[/bold]    {DATASET_NAME}\n'
    f'[bold]Experiment:[/bold] {EXPERIMENT_NAME}',
    title='TR Knowledge QA — Notebook 02',
    border_style='cyan'
))

Working directory set to: /home/coder/eval-agents


2026-03-24 12:05:34,572 WARNING opentelemetry.trace: Overriding of current TracerProvider is not allowed
2026-03-24 12:05:34,626 INFO aieng.agent_evals.langfuse: Langfuse tracing initialized successfully (endpoint: https://us.cloud.langfuse.com/api/public/otel)


╭───────────────────────────────── TR Knowledge QA — Notebook 02 ──────────────────────────────────╮
│ Category:   History                                                                              │
│ Samples:    5                                                                                    │
│ Dataset:    TR-DeepSearchQA                                                                      │
│ Experiment: tr-baseline-v1                                                                       │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

## 2. Upload Dataset to Langfuse

We upload a small subset of DeepSearchQA to Langfuse as a **named dataset**.
This lets us track experiments against the same fixed set of questions.

Re-running this cell is safe — items are deduplicated by content hash.

In [3]:
dataset = DeepSearchQADataset()
examples = dataset.get_by_category(CATEGORY)[:NUM_SAMPLES]

console.print(f'Loaded [cyan]{len(examples)}[/cyan] examples from [bold]{CATEGORY}[/bold]')

# Write to temp JSONL and upload
with tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=False, encoding='utf-8') as f:
    for ex in examples:
        record = {
            'input': ex.problem,
            'expected_output': ex.answer,
            'metadata': {
                'example_id': ex.example_id,
                'category': ex.problem_category,
                'answer_type': ex.answer_type,
            }
        }
        f.write(json.dumps(record, ensure_ascii=False) + '\n')
    temp_path = f.name

await upload_dataset_to_langfuse(dataset_path=temp_path, dataset_name=DATASET_NAME)
os.unlink(temp_path)
console.print(f'[green]✓[/green] Dataset [bold]{DATASET_NAME}[/bold] ready in Langfuse')

2026-03-24 12:05:51,767 INFO aieng.agent_evals.knowledge_qa.data.deepsearchqa: Downloading DeepSearchQA dataset...
2026-03-24 12:05:52,459 INFO aieng.agent_evals.knowledge_qa.data.deepsearchqa: Dropped 4 examples with missing answers
2026-03-24 12:05:52,460 INFO aieng.agent_evals.knowledge_qa.data.deepsearchqa: Loaded 896 examples from DeepSearchQA


Loaded 5 examples from History

2026-03-24 12:05:52,523 INFO aieng.agent_evals.langfuse: Loading dataset from '/tmp/tmput92ekwm.jsonl'


Uploading Langfuse dataset 'TR-DeepSearchQA' ━━━━━━━━━━━━━━━ 5/5 0:00:00 0:00:00m 0:00:00


2026-03-24 12:05:53,729 INFO aieng.agent_evals.langfuse: Uploaded 5 items to dataset 'TR-DeepSearchQA'


✓ Dataset TR-DeepSearchQA ready in Langfuse

## 3. Define Agent Task

The `agent_task` function is called once per dataset item.
It runs the `KnowledgeGroundedAgent` and returns the final answer string.

In [4]:
async def agent_task(*, item: Any, **kwargs: Any) -> str:
    """Run the PlanReAct Knowledge Agent on one dataset item."""
    agent = KnowledgeGroundedAgent(enable_planning=True)
    response = await agent.answer_async(item.input)

    # Attach execution metadata to the Langfuse span for trace-level inspection
    client_manager = AsyncClientManager.get_instance()
    client_manager.langfuse_client.update_current_span(
        metadata=response.model_dump(exclude={'text'})
    )
    return response.text

console.print('[green]✓[/green] agent_task defined')

✓ agent_task defined

## 4. Define the Output-Level Evaluator

The `deepsearchqa_evaluator` is an **LLM-as-judge** using the official DeepSearchQA grading methodology.

For each question it returns four scores:
- `Precision` — correctness of what the agent said
- `Recall` — completeness of the agent's answer
- `F1` — overall answer quality (0.0 to 1.0)
- `Outcome` — one of: `fully_correct`, `correct_with_extraneous`, `partially_correct`, `fully_incorrect`

In [5]:
async def deepsearchqa_evaluator(
    *,
    input: str,
    output: str,
    expected_output: str,
    metadata: dict[str, Any] | None = None,
    **kwargs: Any,
) -> list[Evaluation]:
    """Score the agent's answer against DeepSearchQA ground truth."""
    answer_type = (metadata or {}).get('answer_type', 'Set Answer')
    try:
        result = await evaluate_deepsearchqa_async(
            question=input,
            answer=str(output),
            ground_truth=expected_output,
            answer_type=answer_type,
            model_config=LLMRequestConfig(temperature=0.0),
        )
        return result.to_evaluations()
    except Exception as e:
        return DeepSearchQAResult.error_evaluations(str(e))

console.print('[green]✓[/green] deepsearchqa_evaluator defined')

✓ deepsearchqa_evaluator defined

## 5. Run the Experiment

`run_experiment` runs `agent_task` on every item in the dataset, then calls `deepsearchqa_evaluator`
on each output. Results are stored in Langfuse under the experiment name.

> **Tip:** To run a new experiment (e.g. different model or category), update `EXPERIMENT_NAME` in Cell 1 and re-run from there.

In [8]:
result =  run_experiment(
    DATASET_NAME,
    name=EXPERIMENT_NAME,
    task=agent_task,
    evaluators=[deepsearchqa_evaluator],
    description=f'TR Knowledge QA — {CATEGORY} — {NUM_SAMPLES} samples',
    max_concurrency=1,
)

console.print('[green]✓[/green] Experiment complete!')
if result.dataset_run_url:
    display(HTML(f'<p>View in Langfuse: <a href="{result.dataset_run_url}" target="_blank">{result.dataset_run_url}</a></p>'))

2026-03-24 13:38:21,105 INFO aieng.agent_evals.knowledge_qa.agent: Answering question: According to the Social Security Administration, what are the identical names that show up in the to...
2026-03-24 13:38:21,267 INFO google_adk.google.adk.models.google_llm: Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
2026-03-24 13:38:24,695 INFO google_adk.google.adk.models.google_llm: Response received from the model.
2026-03-24 13:38:24,704 INFO aieng.agent_evals.knowledge_qa.agent: Extracted plan with 5 steps
2026-03-24 13:38:24,705 INFO aieng.agent_evals.knowledge_qa.agent: Tool call: google_search({'query': "Social Security Administration top boy's names 1975 Maine Texas California"})
2026-03-24 13:38:24,820 INFO google_genai.models: AFC is enabled with max remote calls: 10.
2026-03-24 13:38:37,678 INFO aieng.agent_evals.knowledge_qa.agent: Tool response: google_search completed
2026-03-24 13:38:38,181 INFO google_adk.google.adk.models.gemin

✓ Experiment complete!

## 6. Inspect Results

In [9]:
rows = []
for item_result in result.item_results:
    question = str(item_result.item.input)
    row = {'question': question[:60] + '...' if len(question) > 60 else question}
    for evaluation in item_result.evaluations or []:
        row[evaluation.name] = evaluation.value
    rows.append(row)

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Summary means
numeric_cols = [c for c in ['F1', 'Precision', 'Recall'] if c in df.columns]
if numeric_cols:
    means_table = Table(title='Mean Scores Across All Questions')
    means_table.add_column('Metric', style='cyan')
    means_table.add_column('Mean', style='white')
    means_table.add_column('Interpretation', style='dim')
    interpretations = {
        'F1': 'Overall answer quality (0=wrong, 1=perfect)',
        'Precision': 'Fraction of agent claims that were correct',
        'Recall': 'Fraction of correct items the agent found',
    }
    for col in numeric_cols:
        means_table.add_row(col, f'{df[col].mean():.3f}', interpretations.get(col, ''))
    console.print(means_table)

                                                       question           Outcome  F1  Precision  Recall
According to the Social Security Administration, what are th...   Fully Incorrect 0.0        0.0     0.0
Using information from the Council on Tall Buildings and Urb... Partially Correct 0.8        0.8     0.8
Who is the Associate Editor of the SBL Study Bible who is a ...     Fully Correct 1.0        1.0     1.0
Please provide me with a list of fires from the San Francisc...   Fully Incorrect 0.0        0.0     0.0
In the year that featured a volcanic eruption in Iceland cau...     Fully Correct 1.0        1.0     1.0


                 Mean Scores Across All Questions                  
┏━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric    ┃ Mean  ┃ Interpretation                              ┃
┡━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ F1        │ 0.560 │ Overall answer quality (0=wrong, 1=perfect) │
│ Precision │ 0.560 │ Fraction of agent claims that were correct  │
│ Recall    │ 0.560 │ Fraction of correct items the agent found   │
└───────────┴───────┴─────────────────────────────────────────────┘

In [ ]:
# Outcome distribution
if 'Outcome' in df.columns:
    outcome_counts = df['Outcome'].value_counts()
    outcome_table = Table(title='Outcome Distribution')
    outcome_table.add_column('Outcome', style='cyan')
    outcome_table.add_column('Count', style='white', justify='right')
    for outcome, count in outcome_counts.items():
        outcome_table.add_row(str(outcome), str(count))
    console.print(outcome_table)

## 7. Iterate

To run a new experiment:

1. Update `EXPERIMENT_NAME` in Cell 1 (e.g. `'tr-no-planning-v2'`)
2. Optionally change `CATEGORY` or `NUM_SAMPLES`
3. Re-run from Cell 1 down

### Ideas to try:

| Change | How | What to look for |
|--------|-----|------------------|
| Disable planning | `KnowledgeGroundedAgent(enable_planning=False)` | Does F1 drop? Is it faster? |
| Different category | Change `CATEGORY` | Which domains are hardest? |
| More samples | Increase `NUM_SAMPLES` to 10 | Is the mean F1 stable? |

Each run creates a new named experiment in Langfuse — compare runs side-by-side in the **Datasets** tab.

**Next:** Open `03_trajectory_evaluation.ipynb` to add trace-level behavior scoring.

In [ ]:
console.print(Panel(
    '[green]✓[/green] Notebook 02 complete!\n\n'
    '[cyan]Next:[/cyan] Open [bold]03_trajectory_evaluation.ipynb[/bold]\n'
    'to add trace-level behavior scoring on top of these results.',
    title='Done',
    border_style='green',
))